---
title: Single-cell scatter plot
author: Zafer Kosar
format:
    html:
        code-fold: true
        code-summary: "Show code"
---

In [ ]:
# Core scverse libraries

# data manipulation
import polars as pl
import scanpy as sc

# ggplot but interactive and python
from lets_plot import (
    LetsPlot,
    aes,
    arrow,
    element_blank,
    element_rect,
    element_text,
    geom_point,
    geom_segment,
    ggplot,
    ggsize,
    ggtb,
    guide_legend,
    labs,
    layer_tooltips,
    scale_color_continuous,
    scale_color_hue,
    scale_shape,
    theme,
    theme_classic,
)

LetsPlot.setup_html()

import inspect
from typing import Literal

from lets_plot import *
from lets_plot.plot.core import PlotSpec

In [ ]:
# read the sampel data
adata = sc.read("data/pbmc3k_pped.h5ad")

In [ ]:
def theme_scatter(func):
    def modifier(*args, **kwargs):
        plot = func(*args, **kwargs)
        plot += (
            theme_classic()
            + theme(
                axis_text_x=element_blank(),
                axis_text_y=element_blank(),
                axis_ticks_y=element_blank(),
                axis_ticks_x=element_blank(),
                text=element_text(color="#1f1f1f", family="Arial", size=12, face="bold"),
                title=element_text(color="#1f1f1f", family="Arial"),
                axis_title_x=element_text(color="#3f3f3f", family="Arial", size=18),
                axis_title_y=element_text(color="#3f3f3f", family="Arial", size=18),
                legend_text=element_text(color="#1f1f1f", size=10, face="plain"),
                axis_line=element_blank(),
            )
            + ggsize(800, 600)
            + scale_shape(guide=guide_legend(nrow=3))
        )

        if kwargs.get("arrow_axis", True) is True:
            arrow_size = kwargs.get("arrow_size", 0.25)
            plot += theme(
                axis_title_x=element_text(hjust=arrow_size / 2),
                axis_title_y=element_text(hjust=arrow_size / 2),
            )

        return plot

    return modifier

In [ ]:
@theme_scatter
def scatter(
    data,
    key: Literal["leiden", "louvain"] | str = "leiden",
    *,
    dimensions: Literal["umap", "pca", "tsne"] = "umap",
    size: float = 0.8,
    interactive: bool = False,
    color_low: str = "#ffffff",
    color_high: str = "#377eb8",
    arrow_axis: bool = True,
    arrow_size: float = 0.25,
    arrow_color: str = "#3f3f3f",
) -> PlotSpec:
    # Handling Data tpyes
    if not isinstance(data, sc.AnnData):
        raise ValueError("data must be an AnnData object")

    key_col = key

    # get the coordinates of the cells in the dimension reduced space
    frame = pl.from_numpy(
        data.obsm[f"X_{dimensions}"], schema=[f"{dimensions}1", f"{dimensions}2"]
    ).with_columns(pl.Series("ID", data.obs_names))

    # -------------------------- IF IT IS A CLUSTER --------------------------
    if key in ["leiden", "louvain"]:  # if it is a clustering
        key_col = "Cluster"  # update the key column name if it is a cluster
        frame = frame.with_columns(
            pl.Series("ID", data.obs_names), pl.Series(key_col, data.obs[key])
        )
        # cluster scatter
        scttr = (
            ggplot(data=frame)
            + geom_point(
                aes(x=f"{dimensions}2", y=f"{dimensions}1", color=key_col),
                size=size,
                tooltips=layer_tooltips(["ID", key_col]),
            )
            + scale_color_hue()
            + labs(
                x=f"{dimensions}2".upper(), y=f"{dimensions}1".upper()
            )  # UMAP1 and UMAP2 rather than umap1 and umap2 etc.,
            + scale_shape(guide=guide_legend(nrow=3))
        )
    # -------------------------- IF IT IS A GENE --------------------------
    elif key in data.var_names:  # if it is a gene
        # adata.X is a matrix , axis0 is cells, axis1 is genes
        # find the index of the gene
        index = data.var_names.get_indexer(
            data.var_names[data.var_names.str.startswith(key)]
        )  # get the index of the gene
        frame = frame.with_columns(
            pl.Series("ID", data.obs_names),
            pl.Series(key, data.X[:, index].flatten().astype("float64")),
        )
        scttr = (
            ggplot(data=frame)
            + geom_point(
                aes(x=f"{dimensions}2", y=f"{dimensions}1", color=key),
                size=size,
                tooltips=layer_tooltips(["ID", key]),
            )
            + scale_color_continuous(low=color_low, high=color_high)
            + labs(
                x=f"{dimensions}2".upper(), y=f"{dimensions}1".upper()
            )  # UMAP1 and UMAP2 rather than umap1 and umap2 etc.,
        )
    # -------------------------- GEOM SEGMENT --------------------------
    if arrow_axis:
        x_max = frame.select(f"{dimensions}2").max().item()
        x_min = frame.select(f"{dimensions}2").min().item()
        y_max = frame.select(f"{dimensions}1").max().item()
        y_min = frame.select(f"{dimensions}1").min().item()

        # find total difference between the max and min for both axis
        x_diff = x_max - x_min
        y_diff = y_max - y_min

        # find the ends of the arrows
        xend = x_min + arrow_size * x_diff
        yend = y_min + arrow_size * y_diff

        # adjust bottom ends of arrows
        x_adjusted = x_min - x_diff * 0.005
        y_adjusted = y_min - y_diff * 0.005

        # X axis
        scttr += geom_segment(
            x=x_adjusted,
            y=y_min,
            xend=xend,
            yend=y_min,
            color=arrow_color,
            size=3,
            arrow=arrow(20),
        )
        # Y axis
        scttr += geom_segment(
            x=x_min,
            y=y_adjusted,
            xend=x_min,
            yend=yend,
            color=arrow_color,
            size=3,
            arrow=arrow(20),
        )

    # -------------------------- NOT A GENE OR CLUSTER --------------------------
    else:
        msg = f"'{key}' is not present in cluster names nor gene names"
        raise msg

    if interactive:
        return scttr + ggtb()
    else:
        return scttr

In [ ]:
signature = inspect.signature(scatter)

In [ ]:
def get_default_args(func):
    signature = inspect.signature(func)
    return {
        k: v.default
        for k, v in signature.parameters.items()
        if v.default is not inspect.Parameter.empty
    }


get_default_args(scatter)

In [ ]:
print(scatter.__kwdefaults__)

## Test with Cluster [leiden]

In [ ]:
scatter(adata, interactive=True, size=1) + scale_color_brewer(palette="Set2")

In [ ]:
(
    scatter(
        adata,
        interactive=True,
        size=0.8,
    )
    + ggsize(600, 400)
).to_png("scatter.png")

In [ ]:
(
    scatter(
        adata,
        interactive=True,
        size=0.8,
    )
    + ggsize(600, 400)
    + theme(
        plot_background=element_rect(color="red", size=4),
        legend_background=element_rect(color="blue", size=4),
    )
).to_png("scatter_rect.png")

## Testing with a gene

In [ ]:
scatter(adata, "MT-ND2", size=1.5, interactive=True)

### Customize Colors

In [ ]:
scatter(adata, "MT-ND2", size=1.5, interactive=True, color_high="red", color_low="white")

In [ ]:
for i in adata.obsm:
    print(i)

In [ ]:
adata

In [ ]:
adata.obsm["X_umap"].shape

In [ ]:
adata.obsm["X_pca"][:, :2].shape

In [ ]:
adata.varm["PCs"].shape

In [ ]:
adata.uns["pca"]